# Agentic AI Project

## Introduction

Agentic AI is an approach in which an AI system can understand a task, decide what action is required, use available tools, observe the results, and continue the process until the task is completed.

In this project, a Large Language Model (LLM) is connected with two tools: a Calculator and a Word Counter. The agent can decide which tool is required based on the user's request.

The project demonstrates the basic Agentic AI cycle:

**THINK → ACT → OBSERVE → REPEAT → FINAL**

The system also includes input validation, safe calculator execution, error handling, and automated testing to make the agent more reliable and secure.

## Objective

The objectives of this project are:

- To understand the basic working of Agentic AI.
- To connect an LLM with external tools.
- To implement a THINK → ACT → OBSERVE → REPEAT workflow.
- To demonstrate the use of multiple tools in a single request.
- To validate tool arguments before execution.
- To handle invalid inputs and tool errors safely.
- To test the implementation using automated tests.

In [1]:
!pip install -q groq

In [2]:
from google.colab import userdata
from groq import Groq

# Get the API key securely from Colab Secrets
api_key = userdata.get("GROQ_API_KEY")

# Create the Groq client
client = Groq(api_key=api_key)

# Model used by the agent
MODEL = "openai/gpt-oss-120b"

print("Groq connected successfully!")
print("Model:", MODEL)

Groq connected successfully!
Model: openai/gpt-oss-120b


In [3]:
import ast
import operator

# Allowed mathematical operators
OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}


def evaluate_allowed_node(node):
    """Evaluate only approved mathematical AST nodes."""

    # Allow numbers
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    # Allow binary operations: +, -, *, /, **
    if isinstance(node, ast.BinOp) and type(node.op) in OPERATORS:
        left = evaluate_allowed_node(node.left)
        right = evaluate_allowed_node(node.right)

        return OPERATORS[type(node.op)](left, right)

    # Allow negative numbers
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPERATORS:
        operand = evaluate_allowed_node(node.operand)

        return OPERATORS[type(node.op)](operand)

    # Reject everything else
    raise ValueError(
        "Unsupported expression. Only basic mathematical operations are allowed."
    )


def safe_calculator(expression):
    """Safely calculate a mathematical expression."""

    if not isinstance(expression, str):
        return "Error: Expression must be a string."

    if not expression.strip():
        return "Error: Expression cannot be empty."

    try:
        tree = ast.parse(expression, mode="eval")
        result = evaluate_allowed_node(tree.body)
        return str(result)

    except ZeroDivisionError:
        return "Error: Division by zero is not allowed."

    except (SyntaxError, ValueError):
        return "Error: Invalid or unsupported mathematical expression."

    except OverflowError:
        return "Error: Calculation result is too large."


# Basic tests
print("Test 1:", safe_calculator("25 * 4"))
print("Test 2:", safe_calculator("45 + 10 * 2"))
print("Test 3:", safe_calculator("100 / 5"))

Test 1: 100
Test 2: 65
Test 3: 20.0


In [4]:
def word_counter(text):
    """Count the number of words in a text."""

    # Validate the input type
    if not isinstance(text, str):
        return "Error: Text must be a string."

    # Handle empty input
    if not text.strip():
        return "Error: Text cannot be empty."

    # Count words
    return len(text.split())


# Test the Word Counter
print(
    "Word Counter Test:",
    word_counter("Agentic AI can think act observe and repeat.")
)

Word Counter Test: 8


In [5]:
# Tool definitions for the AI

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Safely calculate a mathematical expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression such as 25 * 4."
                }
            },
            "required": ["expression"],
            "additionalProperties": False
        }
    }
}


word_counter_tool = {
    "type": "function",
    "function": {
        "name": "word_counter",
        "description": "Count the number of words in a given text.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The text whose words should be counted."
                }
            },
            "required": ["text"],
            "additionalProperties": False
        }
    }
}


# List of all available tools
all_tools = [
    calculator_tool,
    word_counter_tool
]

print("Two tools are ready:")
print("1. Calculator")
print("2. Word Counter")

Two tools are ready:
1. Calculator
2. Word Counter


In [6]:
# Connect each tool name to its Python function

available_functions = {
    "calculator": safe_calculator,
    "word_counter": word_counter
}

print("Tool functions connected successfully!")

Tool functions connected successfully!


In [7]:
import json


def validate_tool_arguments(function_name, raw_arguments):
    """
    Validate and convert the arguments received from the AI.
    Returns either:
    - (True, arguments)
    - (False, error_message)
    """

    # Check whether the arguments are valid JSON
    try:
        arguments = json.loads(raw_arguments)
    except json.JSONDecodeError as error:
        return False, f"Error: Invalid JSON arguments - {error}"

    # Arguments must be a JSON object
    if not isinstance(arguments, dict):
        return False, "Error: Tool arguments must be a JSON object."

    # Validate Calculator arguments
    if function_name == "calculator":

        if "expression" not in arguments:
            return False, "Error: Calculator requires 'expression'."

        if not isinstance(arguments["expression"], str):
            return False, "Error: Calculator 'expression' must be a string."

        if not arguments["expression"].strip():
            return False, "Error: Calculator expression cannot be empty."

        # Reject unexpected arguments
        if set(arguments.keys()) != {"expression"}:
            return False, "Error: Invalid calculator arguments."

    # Validate Word Counter arguments
    elif function_name == "word_counter":

        if "text" not in arguments:
            return False, "Error: Word Counter requires 'text'."

        if not isinstance(arguments["text"], str):
            return False, "Error: Word Counter 'text' must be a string."

        if not arguments["text"].strip():
            return False, "Error: Word Counter text cannot be empty."

        # Reject unexpected arguments
        if set(arguments.keys()) != {"text"}:
            return False, "Error: Invalid word counter arguments."

    else:
        return False, f"Error: Unknown tool '{function_name}'."

    return True, arguments


print("Tool argument validation is ready!")

Tool argument validation is ready!


In [8]:
def run_agent(user_question, max_iterations=5):

    messages = [
        {
            "role": "system",
            "content": """
You are an Agentic AI assistant.

Follow this process:
1. THINK - understand the user's request.
2. ACT - select an appropriate tool when needed.
3. OBSERVE - examine the tool result.
4. REPEAT - use another tool if more work is required.
5. FINAL - provide the final answer.

Available tools:
- Calculator: safely performs mathematical calculations.
- Word Counter: counts words in text.

Use tools only when they are useful.
"""
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    for iteration in range(max_iterations):

        print(f"\n========== ITERATION {iteration + 1} ==========")
        print("THINK → Agent is deciding what to do...")

        # Ask the AI what to do next
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=all_tools,
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        # If no tool is needed, return the final answer
        if not assistant_message.tool_calls:

            print("FINAL → No more tools are required.")
            return assistant_message.content

        # Process each requested tool
        for tool_call in assistant_message.tool_calls:

            function_name = tool_call.function.name
            raw_arguments = tool_call.function.arguments

            print(f"ACT → Using tool: {function_name}")
            print(f"Arguments → {raw_arguments}")

            # Validate tool arguments
            is_valid, validated_arguments = validate_tool_arguments(
                function_name,
                raw_arguments
            )

            if not is_valid:

                result = validated_arguments
                print("OBSERVE →", result)

            else:

                # Execute the validated tool
                try:
                    result = available_functions[function_name](
                        **validated_arguments
                    )

                except (TypeError, ValueError, ZeroDivisionError) as error:
                    result = f"Error: Tool execution failed - {error}"

                print("OBSERVE → Tool result:", result)

            # Send the tool result back to the AI
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": str(result)
            })

    return "Error: Maximum agent iterations reached."

In [9]:
question = "What is 125 * 8 + 50?"

print("USER →", question)

answer = run_agent(question)

print("\nFINAL ANSWER →", answer)

USER → What is 125 * 8 + 50?

========== ITERATION 1 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: calculator
Arguments → {"expression":"125 * 8 + 50"}
OBSERVE → Tool result: 1050

========== ITERATION 2 ==========
THINK → Agent is deciding what to do...
FINAL → No more tools are required.

FINAL ANSWER → The result of \(125 \times 8 + 50\) is **1050**.


In [10]:
question = "How many words are in this sentence: Agentic AI can think, act, observe, and repeat."

print("USER →", question)

answer = run_agent(question)

print("\nFINAL ANSWER →", answer)

USER → How many words are in this sentence: Agentic AI can think, act, observe, and repeat.

========== ITERATION 1 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: word_counter
Arguments → {"text":"Agentic AI can think, act, observe, and repeat."}
OBSERVE → Tool result: 8

========== ITERATION 2 ==========
THINK → Agent is deciding what to do...
FINAL → No more tools are required.

FINAL ANSWER → The sentence contains **8 words**.


In [12]:
# Combined multi-tool agent test

question = """
Calculate 125 * 8 + 50 and also count the words in this sentence:
Agentic AI can think, act, observe, and repeat.
"""

print("USER →", question)

answer = run_agent(question)

print("\nFINAL ANSWER →", answer)

USER → 
Calculate 125 * 8 + 50 and also count the words in this sentence:
Agentic AI can think, act, observe, and repeat.


========== ITERATION 1 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: calculator
Arguments → {"expression":"125 * 8 + 50"}
OBSERVE → Tool result: 1050

========== ITERATION 2 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: word_counter
Arguments → {"text":"Agentic AI can think, act, observe, and repeat."}
OBSERVE → Tool result: 8

========== ITERATION 3 ==========
THINK → Agent is deciding what to do...
FINAL → No more tools are required.

FINAL ANSWER → **Calculation:**  
\(125 \times 8 + 50 = 1050\)

**Word count:**  
The sentence “Agentic AI can think, act, observe, and repeat.” contains **8 words**.


In [11]:
# Automated tests for the Agentic AI project

def test_calculator():
    # Normal calculations
    assert safe_calculator("25 * 4") == "100"
    assert safe_calculator("45 + 10 * 2") == "65"
    assert safe_calculator("100 / 5") == "20.0"
    assert safe_calculator("2 ** 3") == "8"

    # Unsafe expression
    unsafe_result = safe_calculator("__import__('os').system('echo unsafe')")
    assert unsafe_result.startswith("Error:")

    # Division by zero
    assert safe_calculator("10 / 0") == "Error: Division by zero is not allowed."

    # Empty expression
    assert safe_calculator("") == "Error: Expression cannot be empty."

    print("✓ Calculator tests passed")


def test_word_counter():
    # Normal input
    assert word_counter("Agentic AI is useful.") == 4
    assert word_counter("Hello world") == 2

    # Invalid inputs
    assert word_counter("") == "Error: Text cannot be empty."
    assert word_counter("   ") == "Error: Text cannot be empty."

    print("✓ Word Counter tests passed")


def test_argument_validation():

    # Valid calculator arguments
    valid, arguments = validate_tool_arguments(
        "calculator",
        '{"expression": "25 * 4"}'
    )

    assert valid is True
    assert arguments["expression"] == "25 * 4"

    # Malformed JSON
    valid, error = validate_tool_arguments(
        "calculator",
        '{"expression": '
    )

    assert valid is False
    assert error.startswith("Error:")

    # Missing argument
    valid, error = validate_tool_arguments(
        "calculator",
        '{}'
    )

    assert valid is False
    assert error.startswith("Error:")

    # Wrong argument type
    valid, error = validate_tool_arguments(
        "calculator",
        '{"expression": 100}'
    )

    assert valid is False
    assert error.startswith("Error:")

    # Unknown tool
    valid, error = validate_tool_arguments(
        "unknown_tool",
        '{}'
    )

    assert valid is False
    assert error.startswith("Error:")

    # Extra argument
    valid, error = validate_tool_arguments(
        "calculator",
        '{"expression": "25 * 4", "extra": "invalid"}'
    )

    assert valid is False
    assert error.startswith("Error:")

    print("✓ Argument validation tests passed")


# Run all tests
test_calculator()
test_word_counter()
test_argument_validation()

print("\nAll automated tests passed successfully!")

✓ Calculator tests passed
✓ Word Counter tests passed
✓ Argument validation tests passed

All automated tests passed successfully!


# Agentic AI Architecture

```text
                    ┌───────────────────┐
                    │    User Request   │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │   Groq LLM Agent  │
                    │      THINK        │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │   Tool Selection  │
                    └─────────┬─────────┘
                              │
                 ┌────────────┴────────────┐
                 │                         │
                 ▼                         ▼
        ┌─────────────────┐       ┌─────────────────┐
        │  Calculator     │       │  Word Counter   │
        │  Safe AST       │       │  Text Validation│
        └────────┬────────┘       └────────┬────────┘
                 │                         │
                 └────────────┬────────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │     OBSERVE       │
                    │   Tool Result     │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │   Groq LLM Agent  │
                    │ REPEAT / FINAL    │
                    └─────────┬─────────┘
                              │
                              ▼
                    ┌───────────────────┐
                    │    Final Answer   │
                    └───────────────────┘

# Project Documentation

## 1. Objective

The main objective of this project is to demonstrate how an Agentic AI system can use a Large Language Model (LLM) together with external tools to solve user tasks.

The project focuses on:

- Understanding the Agentic AI workflow.
- Connecting an LLM with external tools.
- Using multiple tools in a single request.
- Validating tool arguments before execution.
- Handling invalid inputs safely.
- Implementing automated tests.
- Demonstrating the THINK → ACT → OBSERVE → REPEAT process.

## 2. Tools Used

### Calculator

The Calculator tool performs mathematical calculations using a safe AST-based approach. Only approved mathematical operations are allowed.

Supported operations include:

- Addition
- Subtraction
- Multiplication
- Division
- Power
- Unary negative values

Unsafe expressions are rejected instead of being executed.

### Word Counter

The Word Counter tool counts the number of words in a given text.

It also validates the input and returns an error when the text is empty or invalid.

## 3. Agent Workflow

The agent follows the following process:

**THINK → ACT → OBSERVE → REPEAT → FINAL**

### THINK
The LLM understands the user's request and determines what action is required.

### ACT
The agent selects and calls the appropriate tool.

### OBSERVE
The result returned by the tool is examined by the agent.

### REPEAT
If the request contains additional tasks, the agent can select another tool and continue the process.

### FINAL
After completing the required tasks, the agent generates the final response.

## 4. Security and Validation

The project includes several safety mechanisms:

- AST-based safe calculator execution.
- JSON argument validation.
- Required argument checking.
- Data type validation.
- Empty input validation.
- Unknown tool detection.
- Extra argument detection.
- Explicit error messages.
- Controlled maximum agent iterations.
- No unsafe Python expression execution.

## 5. Multi-Tool Capability

The project demonstrates that the agent can handle multiple tasks in a single user request.

For example, a user can ask the agent to:

1. Calculate a mathematical expression.
2. Count the words in a sentence.

The agent can select the Calculator first, observe its result, then select the Word Counter and finally provide both results.

This demonstrates the **REPEAT** stage of the Agentic AI workflow.

## 6. Testing

Automated tests are included to verify:

- Normal calculator operations.
- Unsafe calculator expressions.
- Division by zero.
- Empty calculator input.
- Normal word counting.
- Empty word-counter input.
- Valid JSON arguments.
- Malformed JSON.
- Missing arguments.
- Incorrect argument types.
- Unknown tools.
- Extra arguments.

All tests are executed before the final project is completed.

## 7. Technologies Used

- Python
- Google Colab
- Groq API
- Large Language Model
- Python AST
- JSON
- Tool Calling

## 8. Expected Outcome

The completed system can understand user requests, select appropriate tools, execute them safely, observe the results, and generate a final response.

The project therefore provides a practical demonstration of how external tools can be integrated with an LLM to build a basic Agentic AI system.

# Conclusion

This project demonstrates the basic working of an Agentic AI system using a Large Language Model and external tools.

The agent can understand a user request, select the required tool, execute the task safely, observe the result, and continue the process when multiple tasks are given.

The project includes two tools:

- **Calculator** – performs mathematical calculations using safe AST-based processing.
- **Word Counter** – counts the number of words in a given text.

The combined multi-tool example demonstrates how the agent can handle more than one task within a single user request.

The implementation also includes:

- Input and argument validation
- Safe tool execution
- Explicit error handling
- Automated testing
- Agent iteration limits
- Architecture documentation

Overall, the project demonstrates the **THINK → ACT → OBSERVE → REPEAT → FINAL** workflow and provides a simple practical example of Agentic AI.


In [13]:
%%writefile requirements.txt
groq


Overwriting requirements.txt
